# Sequence Parallelism Tutorial

## Overview

Sequence parallelism splits the sequence dimension across GPUs, reducing activation memory for long sequences.

### Learning Objectives
- Understand sequence parallel architecture
- Combine with tensor parallelism
- Analyze memory savings

### References
- Korthikanti et al., "Reducing Activation Recomputation in Large Transformer Models", MLSys 2023

## 1. Mathematical Foundation

### Activation Memory

For a transformer layer with batch $B$, sequence length $S$, hidden size $H$:

**Without Sequence Parallelism:**
$$M_{\text{act}} = O(B \times S \times H)$$

**With Sequence Parallelism (N GPUs):**
$$M_{\text{act}} = O(B \times \frac{S}{N} \times H)$$

### Architecture
```
Input [B, S, H]
    │
    ▼ Split sequence
┌───┴───┬───┴───┐
│GPU 0  │GPU 1  │  Each: [B, S/N, H]
│LayerNorm      │
│Dropout        │
└───┬───┴───┬───┘
    │ AllGather (for attention)
    ▼
  Attention [B, S, H]
    │
    ▼ ReduceScatter
┌───┴───┬───┴───┐
│GPU 0  │GPU 1  │  Each: [B, S/N, H]
└───────┴───────┘
```

In [ ]:
import torch
import torch.nn as nn
import torch.distributed as dist

class SequenceParallelLayerNorm(nn.Module):
    """LayerNorm with sequence parallelism.
    
    Each GPU processes a portion of the sequence.
    """
    def __init__(self, hidden_size: int):
        super().__init__()
        self.norm = nn.LayerNorm(hidden_size)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, S/N, H] - already split across sequence
        return self.norm(x)

def calculate_memory_savings(batch: int, seq_len: int, hidden: int, num_gpus: int):
    """Calculate activation memory savings."""
    baseline = batch * seq_len * hidden * 2  # FP16
    with_sp = batch * (seq_len // num_gpus) * hidden * 2
    
    print(f"Config: B={batch}, S={seq_len}, H={hidden}, GPUs={num_gpus}")
    print(f"Baseline: {baseline/1e9:.2f} GB")
    print(f"With SP: {with_sp/1e9:.2f} GB")
    print(f"Savings: {(1-with_sp/baseline)*100:.1f}%")

calculate_memory_savings(8, 8192, 4096, 8)

## 2. Summary

| Feature | Tensor Parallel | Sequence Parallel |
|---------|-----------------|-------------------|
| Splits | Model weights | Activations |
| Memory saved | Parameters | Activations |
| Best for | Large models | Long sequences |
| Communication | AllReduce | AllGather/ReduceScatter |